In [1]:
import pandas as pd
import numpy as np

churn_df = pd.read_pickle("Output/churn_df.pkl")
future_features = pd.read_pickle("Output/future_features.pkl")
true_future_predictions = np.load("Output/true_future_predictions.npy")

In [2]:
master_df = pd.read_pickle("Dataset/Cleaned_data/master_df.pkl")
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 36 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CustomerID             52924 non-null  str           
 1   Transaction_ID         52924 non-null  str           
 2   Transaction_Date       52924 non-null  datetime64[us]
 3   Month                  52924 non-null  str           
 4   Date                   52924 non-null  object        
 5   Week                   52924 non-null  str           
 6   Product_SKU            52924 non-null  str           
 7   Product_Description    52924 non-null  str           
 8   Product_Category       52924 non-null  str           
 9   ABC                    52924 non-null  str           
 10  Quantity               52924 non-null  int64         
 11  Avg_Price              52924 non-null  float64       
 12  Delivery_Charges       52924 non-null  float64       
 13  R

In [3]:
# ensure CustomerID columns have the same dtype
print(churn_df['CustomerID'].dtype, future_features['CustomerID'].dtype)

churn_df['CustomerID'] = churn_df['CustomerID'].astype(str)
future_features['CustomerID'] = future_features['CustomerID'].astype(str)

str int64


In [4]:
churn_df.head()

,CustomerID,frequency_cal,monetary_sum,total_quantity,unique_categories,avg_discount,coupon_usage_rate,total_delivery_fee,total_offline,total_online,...,Location_Washington Dc,KMeans_Label_1,KMeans_Label_2,KMeans_Label_3,future_orders,Churn,Churn_Probability,Churn_Segment,Revenue_At_Risk,Retention_Priority
98,12748,34,78689.99626,4869,17,0.068633,0.362590,8510.90,2059500,1182404.71,...,False,False,False,True,0.0,1,0.800,High Risk,62951.997008,62951.997008
586,15311,16,68113.98480,3598,16,0.063876,0.330144,3505.65,1194500,845752.98,...,False,False,False,True,91.0,0,0.485,Medium Risk,33035.282628,39642.339154
1113,17850,10,37377.51564,1133,15,0.030303,0.306397,3162.62,793000,390789.28,...,False,False,False,True,0.0,1,0.830,High Risk,31023.337981,31023.337981
453,14667,7,28865.82314,1573,13,0.056548,0.351190,1334.58,521500,278569.15,...,False,False,False,False,0.0,1,0.825,High Risk,23814.304090,23814.304090
595,15351,2,23074.43632,2160,8,0.052830,0.264151,558.37,159000,48094.06,...,True,False,False,False,0.0,1,0.830,High Risk,19151.782146,19151.782146


In [5]:
# Build customer intelligence table from churn + CLV outputs

customer_table = churn_df[[
    'CustomerID',
    'Churn_Segment',
    'Churn_Probability',
    'frequency_cal',
    'monetary_sum'
]].copy()

future_clv_df = pd.DataFrame({
    'CustomerID': future_features['CustomerID'],
    'Predicted_CLV': true_future_predictions
})

customer_table = customer_table.merge(
    future_clv_df,
    on='CustomerID',
    how='left'
)

customer_table = customer_table.rename(columns={
    'CustomerID': 'Customer_ID',
    'Churn_Segment': 'Segment',
    'frequency_cal': 'Frequency',
    'monetary_sum': 'Monetary'
})

customer_table = customer_table[
    ['Customer_ID', 'Segment', 'Predicted_CLV',
     'Churn_Probability', 'Frequency', 'Monetary']
]

customer_table.head()

,Customer_ID,Segment,Predicted_CLV,Churn_Probability,Frequency,Monetary
0,12748,High Risk,3753.424585,0.800,34,78689.99626
1,15311,Medium Risk,4379.451142,0.485,16,68113.98480
2,17850,High Risk,1969.182284,0.830,10,37377.51564
3,14667,High Risk,1492.683331,0.825,7,28865.82314
4,15351,High Risk,1840.148750,0.830,2,23074.43632


### Revenue at Risk Analysis

If anny customer has high CLV  and high churn probability  → Revenue at Risk

Current customer churn behavior puts approximately ${...} in future revenue at risk.

In [6]:
# =========================================================
# REVENUE AT RISK
# =========================================================

customer_table["Revenue_At_Risk"] = (
    customer_table["Predicted_CLV"] *
    customer_table["Churn_Probability"]
)

total_revenue_risk = customer_table["Revenue_At_Risk"].sum()

print(
    f"Estimated Revenue At Risk: "
    f"${total_revenue_risk:,.2f}"
)

Estimated Revenue At Risk: $563,768.75


## Strategic Customer Classification

Segmentation: 4 business groups.

In [7]:
# CLV threshold for top 20% customers
clv_threshold = customer_table["Predicted_CLV"].median()
print(f"CLV threshold for top 20% customers: ${clv_threshold:.2f}")

CLV threshold for top 20% customers: $396.81


In [8]:
# =========================================================
# CUSTOMER STRATEGY CLASSIFICATION
# =========================================================

def assign_strategy(row):
    
    segment = row["Segment"]
    revenue_at_risk = row["Revenue_At_Risk"]
    clv = row["Predicted_CLV"]
    
    # Prioritize by segment + value
    if segment == "High Risk":
        if clv >= clv_threshold:
            return "Emergency Intervention"
        else:
            return "High-Risk Automation"
    
    elif segment == "Medium Risk":
        if clv >= clv_threshold:
            return "Critical Retention"
        else:
            return "Targeted Campaign"
    
    else:  # Safe Customer
        if clv >= clv_threshold:
            return "Loyal Growth"
        else:
            return "Nurture"


customer_table["Strategy_Group"] = customer_table.apply(
    assign_strategy,
    axis=1
)

In [9]:
display(customer_table.head())
display(customer_table.describe().T)

,Customer_ID,Segment,Predicted_CLV,Churn_Probability,Frequency,Monetary,Revenue_At_Risk,Strategy_Group
0,12748,High Risk,3753.424585,0.800,34,78689.99626,3002.739668,Emergency Intervention
1,15311,Medium Risk,4379.451142,0.485,16,68113.98480,2124.033804,Critical Retention
2,17850,High Risk,1969.182284,0.830,10,37377.51564,1634.421296,Emergency Intervention
3,14667,High Risk,1492.683331,0.825,7,28865.82314,1231.463748,Emergency Intervention
4,15351,High Risk,1840.148750,0.830,2,23074.43632,1527.323463,Emergency Intervention


,count,mean,std,min,25%,50%,75%,max
Predicted_CLV,1211.0,741.780646,839.782540,0.000,0.00000,396.811276,1430.467268,5046.976425
Churn_Probability,1211.0,0.760983,0.327923,0.005,0.64250,0.940000,0.995000,1.000000
Frequency,1211.0,2.020644,1.944368,1.000,1.00000,1.000000,2.000000,34.000000
Monetary,1211.0,2805.092560,4706.181005,1.320,561.27535,1538.099800,3393.282640,78689.996260
Revenue_At_Risk,1211.0,465.539846,632.005457,0.000,0.00000,55.958640,1041.999576,3002.739668


In [10]:
customer_table.to_pickle("Output/customer_table.pkl")